In [0]:
dbutils.widgets.text("p_batch_id","")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run /Workspace/Users/chrknov6@hotmail.com/formula1/Incremental/00.Configurations

In [0]:
%run "/Workspace/Users/chrknov6@hotmail.com/formula1/Incremental/003.gold helper functions"

In [0]:
from pyspark.sql.functions import col,lit,when

In [0]:
silver_results_table = f'{catalog}.{silver_schema}.results'
silver_sprints_table = f'{catalog}.{silver_schema}.sprints'
gold_table = f'{catalog}.{gold_schema}.fact_session_results'

In [0]:
fact_session_results_df =  (
                              spark.table(silver_results_table)
                                   .filter(col("batch_id") == lit(v_batch_id))
                                   .withColumn("session_type",lit("race"))
                                   .unionByName(spark.table(silver_sprints_table).filter(col("batch_id") == lit(v_batch_id)).withColumn("session_type",lit("sprint")))
                                   .drop(col("race_name"),col("race_date"),col("ingestion_time"),col("filename"))
                                   .withColumn("is_win",when(col("finish_position")== 1,True).otherwise(False))
                                   .withColumn("is_podium",when(col("finish_position") <= 3,True).otherwise(False))
                                   .withColumn("has_points",when(col("points")> 0,True).otherwise(False))
                                   .drop(["batch_id","created_timestamp","updated_timestamp"])
)

In [0]:
write_to_gold(
    input_df=fact_session_results_df,
    table_name=gold_table,
    merge_condition="t.season = s.season and t.round = s.round and t.driver_id = s.driver_id and t.constructor_id = s.constructor_id and t.session_type = s.session_type",
    columns_to_update=["finish_position","finish_position_text","grid_position","completed_laps","car_number","points","status","is_win","is_podium","has_points"]
)